# Building the Model - Fantasy Hockey Weekly Predictions

This notebook builds and explains the base statistical model (v1) used for
weekly predictions in this project. It's meant to be read, not just run,
each section explains why a choice was made, not just what the code does.

What this model does: estimates a player's expected fantasy performance and
how consistent that performance is, then uses both to compute a probability
that one player (or goalie) will outperform another in a given week.

What this model does not do: predict which team wins an actual NHL game. :
See METHODOLOGY.md in the repo for the full write-up.

In [ ]:
#Main packages used
import numpy as np
import pandas as pd
from scipy.stats import norm

import json
from google.colab import files

import requests
import time

import os
from datetime import datetime, timedelta

import random


pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", None)

## Scoring configuration

Fantasy stat categories differ for skaters (forwards/defensemen) vs.
goaltenders, and a custom scoring upload should only be allowed to use
recognized categories - otherwise a typo or unsupported stat would
silently produce wrong fantasy point totals downstream with no warning.

Below, the default scoring config is set to my own league's actual rules.
An optional custom upload is checked against a fixed list of valid Yahoo
fantasy hockey stat abbreviations (split into skater and goalie categories)
before it's allowed to override anything. Any unrecognized key is flagged
and dropped rather than silently accepted. The abbreviations used here are:

**Player Stats**
*   **G**: Goals
*   **A**: Assists
*   **P**: Points
*   **+/-**: Plus/Minus Rating
*   **PIM**: Penalty Minutes
*   **PPG**: Powerplay Goals
*   **PPA**: Powerplay Assists
*   **PPP**: Powerplay Points
*   **SHG**: Shorthanded Goals
*   **SHA**: Shorthanded Assists
*   **SHP**: Shorthanded Points
*   **GWG**: Game-Winning Goals
*   **SOG**: Shots on Goal
*   **SH%**: Shooting Percentage
*   **FW**: Faceoffs Won
*   **FL**: Faceoffs Lost
*   **HIT**: Hits
*   **BLK**: Blocks

**Goalie Stats**
*   **GS**: Games Started
*   **W**: Wins
*   **L**: Losses
*   **SHO**: Shutouts
*   **SA**: Shots Against
*   **SV**: Saves
*   **GA**: Goals Against
*   **GAA**: Goals Against Average
*   **SV%**: Save Percentage


*Note: FW, FL are not currently scoreable, only a faceoff percentage is
available from this data source, not raw win/loss counts, see
future_directions for details.*



In [ ]:
# Recognized fantasy stat categories (skaters and goalies use different sets).
# Based on standard Yahoo Fantasy Hockey abbreviations.

VALID_SKATER_STATS = {
    "G", "A", "P", "+/-", "PIM", "PPG", "PPA", "PPP",
    "SHG", "SHA", "SHP", "GWG", "SOG", "SH%", "FW", "FL", "HIT", "BLK"
}

VALID_GOALIE_STATS = {
    "GS", "W", "L", "SHO", "SA", "SV", "GA", "GAA", "SV%"
}

# My "main" league's actual scoring - used as the default.
default_skater_scoring = {
    "G": 2,
    "A": 1,
    "PIM": -0.5,
    "PPG": 0.5,
    "SHG": 2,
    "SHA": 1,
    "GWG": 1,
    "SOG": 0.1,
    "HIT": 0.25,
    "BLK": 0.25,
}

default_goalie_scoring = {
    "W": 5,
    "GA": -1,
    "SV": 0.1,
    "SHO": 5,
}

skater_scoring = default_skater_scoring
goalie_scoring = default_goalie_scoring

In [ ]:
def validate_scoring_config(config, valid_stats, label):
    """
    Checks that every key in a scoring config is a recognized fantasy stat.
    Returns (clean_config, rejected_keys) - invalid keys are dropped, not silently kept.
    """
    clean_config = {}
    rejected = []

    for stat, value in config.items():
        if stat in valid_stats:
            clean_config[stat] = value
        else:
            rejected.append(stat)

    if rejected:
        print(f"Warning: the following {label} keys were not recognized and were dropped: {rejected}")

    return clean_config, rejected

# Sanity check against our own defaults, should show zero rejections
skater_scoring, _ = validate_scoring_config(skater_scoring, VALID_SKATER_STATS, "skater")
goalie_scoring, _ = validate_scoring_config(goalie_scoring, VALID_GOALIE_STATS, "goalie")

print("Skater scoring:", skater_scoring)
print("Goalie scoring:", goalie_scoring)

In [ ]:
#If you want to import a custom player scoring file.
print("Upload a custom skater scoring config (JSON), or skip this cell to keep the default.")
uploaded = files.upload()

if uploaded:
    custom_filename = list(uploaded.keys())[0]
    with open(custom_filename) as f:
        candidate_config = json.load(f)
    skater_scoring, rejected = validate_scoring_config(candidate_config, VALID_SKATER_STATS, "skater")
    print(f"Custom skater scoring loaded from {custom_filename}: {skater_scoring}")
print("Skater scoring:", skater_scoring)

In [ ]:
#If you want to import a custom goalie scoring file.
print("Upload a custom goalie scoring config (JSON), or skip this cell to keep the default.")
uploaded = files.upload()

if uploaded:
    custom_filename = list(uploaded.keys())[0]
    with open(custom_filename) as f:
        candidate_goalie_config = json.load(f)
    goalie_scoring, rejected = validate_scoring_config(candidate_goalie_config, VALID_GOALIE_STATS, "goalie")
    print(f"Custom goalie scoring loaded from {custom_filename}: {goalie_scoring}")
print("Goalie scoring:", goalie_scoring)

## Section 1, getting raw per-game data

The model needs a per-game stat line for each player, goals, assists,
shots, and so on. In the live version of this project, this comes from
the NHL's public API. For this notebook, I'm using a small hand-built
sample dataset for two players so the logic below is easy to follow and
reproduce without needing a live connection, the actual data pulling
functions get built out later in this notebook.

The two example players below are deliberately chosen to illustrate the
point of this whole model, Player A is a steady, consistent scorer,
Player B has the same average production, but is far more boom-or-bust.
A model that only looked at averages would treat them as identical.

## Where the data comes from

Stats in this project come from the NHL's public web API
(api-web.nhle.com), specifically the player game log endpoint. It's
free, official, and doesn't need an API key, which matters since this
whole project is meant to run on open data anyone can access.

How far back it goes, technically the API has structured data going back
decades, even to the early 1900s for basic schedule info. But detailed,
game by game player stats are really only reliable from around the mid
2000s onward, once the league started tracking things more consistently.

For this model though, we don't need most of that history. What we
actually pull is the last season or two, used as the prior for the
shrinkage estimate, plus the current season, updated week to week as
games are played. So in practice, this notebook will mostly be working
with the last 1 to 2 seasons of data, not the full historical archive.
That's intentional, older data is less relevant to how a player is
performing right now anyway.

One thing worth flagging honestly, this is an unofficial, undocumented
API in the sense that the NHL doesn't publish a formal spec for it. It's
widely used by hobby projects and has been stable for a while, but it
could change or break without notice. If that happens, the fix is just
updating the pull functions in this notebook, the rest of the model
doesn't care where the numbers come from.

## Section 2, building a full player roster

Rather than looking players up one at a time by name, it's more useful
to pull a complete table of every active player, across all 32 teams,
once. That gives us player ids, names, positions, and teams all in one
place, which is what we need for things like scanning a whole roster,
or later, finding sleeper picks league wide.

Goalies show up in this same roster pull, but worth flagging now, once
we get to actual game logs, goalies need separate handling. Their stat
categories are completely different, saves, goals against, decisions,
not goals and assists, so a goalie's game log comes back from the API
with different fields entirely. We'll build a separate goalie pull
function for that reason, not just filter the skater one.

In [ ]:
NHL_TEAMS = [
    "ANA","BOS","BUF","CAR","CBJ","CGY","CHI","COL","DAL","DET",
    "EDM","FLA","LAK","MIN","MTL","NJD","NSH","NYI","NYR","OTT",
    "PHI","PIT","SEA","SJS","STL","TBL","TOR","UTA","VAN","VGK",
    "WPG","WSH"
]

def get_team_roster(team_abbr, retries=3, delay=1):
    """
    Pulls the current roster for one team, returns a DataFrame with
    id, name, and position for every player, skaters and goalies together.
    Retries a couple times with a growing delay if rate limited.
    """
    url = f"https://api-web.nhle.com/v1/roster/{team_abbr}/current"

    for attempt in range(retries):
        response = requests.get(url)
        if response.status_code == 429:
            wait = delay * (attempt + 1)
            print(f"Rate limited on {team_abbr}, waiting {wait}s and retrying")
            time.sleep(wait)
            continue
        response.raise_for_status()
        data = response.json()

        rows = []
        for group in ["forwards", "defensemen", "goalies"]:
            for player in data.get(group, []):
                rows.append({
                    "player_id": player["id"],
                    "name": f"{player['firstName']['default']} {player['lastName']['default']}",
                    "position": player["positionCode"],
                    "team": team_abbr,
                })
        return pd.DataFrame(rows)

    raise RuntimeError(f"Failed to pull roster for {team_abbr} after {retries} attempts")

def get_all_players(pause=0.5):
    """
    Loops over every team and builds one big table of all active players.
    Pauses briefly between calls to stay under the API's rate limit.
    """
    all_rosters = []
    for team in NHL_TEAMS:
        roster = get_team_roster(team)
        all_rosters.append(roster)
        time.sleep(pause)
    return pd.concat(all_rosters, ignore_index=True)

all_players = get_all_players()
all_players.tail()


## Section 3, checking the roster pull actually worked

Before trusting all_players for anything downstream, worth running a
few basic checks, not a full test suite, just enough to catch obvious
problems, like a team silently failing, or goalies missing entirely ☹.

In [ ]:
def check_roster_data(players_df):
    checks_passed = 0
    checks_total = 0

    def check(condition, description):
        nonlocal checks_passed, checks_total
        checks_total += 1
        if condition:
            checks_passed += 1
            print(f"[pass] {description}")
        else:
            print(f"[FAIL] {description}")

    # basic shape
    check(len(players_df) > 0, "players_df is not empty")
    check(set(["player_id", "name", "position", "team"]).issubset(players_df.columns),
          "has the expected columns")

    # every team is represented, nothing silently dropped
    missing_teams = set(NHL_TEAMS) - set(players_df["team"].unique())
    check(len(missing_teams) == 0, f"all {len(NHL_TEAMS)} teams are present")
    if missing_teams:
        print(f"       missing teams: {missing_teams}")

    # goalies specifically made it in
    goalie_count = (players_df["position"] == "G").sum()
    check(goalie_count > 0, f"goalies are present ({goalie_count} found)")
    check(goalie_count >= 32 * 2, "roughly the expected number of goalies (at least 2 per team)")

    # no duplicate player ids, would suggest something got pulled twice
    check(players_df["player_id"].is_unique, "no duplicate player ids")

    # rough sanity check on total size, NHL rosters are usually 20-23 active players
    expected_min = 32 * 30
    expected_max = 32 * 55
    check(expected_min <= len(players_df) <= expected_max,
          f"total player count ({len(players_df)}) is in a sane range")

    print(f"\n{checks_passed}/{checks_total} checks passed")

check_roster_data(all_players)

## Section 4, caching and cleaning

Two different caching needs here. The player roster barely changes day
to day, so it just needs an occasional refresh, once a week is plenty.
Game logs are different, during the season they change every time a
game is played, so those need to be checked more often, but still not
refetched on literally every run of the notebook.

The rule, roughly, if the cached file is older than some threshold,
pull fresh data and overwrite it, otherwise just read what's already
saved. Past season data isn't handled yet, that comes back once we get
to the shrinkage section, since that's the first place it's actually
needed.

In [ ]:
def fetch_game_log(player_id, season, game_type=2):
    """
    Pulls a player's game log for a given season.
    season format is like 20232024, game_type 2 is regular season, 3 is playoffs.
    Returns a DataFrame, one row per game.
    """
    url = f"https://api-web.nhle.com/v1/player/{player_id}/game-log/{season}/{game_type}"
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()

    games = data.get("gameLog", [])
    return pd.DataFrame(games)

In [ ]:
def get_current_season():
    """
    Works out the current NHL season string, like 20262027, based on today's date.
    New seasons typically start in October, so anything July or later counts as
    the start of a new season year.
    """
    today = datetime.now()
    start_year = today.year if today.month >= 7 else today.year - 1
    return f"{start_year}{start_year + 1}"

CURRENT_SEASON = get_current_season()
CURRENT_SEASON
LAST_SEASON = "20252026"

In [ ]:
def get_game_log(player_id, season=CURRENT_SEASON, game_type=2, refresh_after_hours=6, force_refresh=False):
    key = f"gamelog_{player_id}_{season}"
    age = cache_age_hours(key)

    if not force_refresh and age is not None and age < refresh_after_hours:
        print(f"Loading game log for {player_id}, {season} from cache, {age:.1f} hours old")
        return load_from_cache(key)

    print(f"Pulling fresh game log for {player_id}, {season}")
    log_df = fetch_game_log(player_id, season, game_type)

    if log_df.empty:
        print(f"No games found for {player_id} in {season}, season may not have started yet")
        return log_df

    save_to_cache(log_df, key)
    return log_df

In [ ]:
CACHE_DIR = "data"
os.makedirs(CACHE_DIR, exist_ok=True)

def cache_path(key):
    return os.path.join(CACHE_DIR, f"{key}.csv")

def cache_age_hours(key):
    """Returns how old a cached file is, in hours, or None if it doesn't exist yet."""
    path = cache_path(key)
    if not os.path.exists(path):
        return None
    modified = datetime.fromtimestamp(os.path.getmtime(path))
    return (datetime.now() - modified).total_seconds() / 3600

def save_to_cache(df, key):
    if df.empty:
        print(f"Not caching '{key}', the data is empty (nothing to save yet)")
        return
    df.to_csv(cache_path(key), index=False)

def load_from_cache(key):
    return pd.read_csv(cache_path(key))

In [ ]:
def get_current_players(refresh_after_hours=168, force_refresh=False):
    """
    Loads the full player roster, from cache if it's fresh enough,
    otherwise pulls fresh and re-caches. Default refresh window is a
    week (168 hours), since rosters don't change often.
    """
    age = cache_age_hours("all_players")

    if not force_refresh and age is not None and age < refresh_after_hours:
        print(f"Loading all_players from cache, {age:.1f} hours old")
        return load_from_cache("all_players")

    print("Pulling fresh player roster data")
    players_df = get_all_players()
    save_to_cache(players_df, "all_players")
    return players_df

all_players = get_current_players()

In [ ]:
def lookup_player_id(name, players_df=all_players):
    """
    Looks up a player's id from the all_players table by name.
    Simple substring match, case insensitive, so partial names work too.
    """
    matches = players_df[players_df["name"].str.contains(name, case=False)]

    if matches.empty:
        print(f"No player found matching '{name}'")
        return None

    if len(matches) > 1:
        print(f"Multiple matches for '{name}', returning the first one:")
        print(matches[["name", "team", "position"]])

    return matches.iloc[0]["player_id"]

## Cleaning the game log

A few things worth fixing before this data feeds into any stats,
missing values, in case a game came back incomplete, columns that
should be numbers but arrive as text, and turning the date column into
an actual date, mostly so we can sort correctly and check "how old is
this data" downstream.

In [ ]:
def clean_game_log(log_df, numeric_columns, required_columns=None):
    """
    Cleans a raw game log. numeric_columns get coerced to numeric types.
    required_columns (defaults to numeric_columns if not given) get checked
    for missing values, but aren't forced to be numeric, this matters for
    columns like 'decision', which are categorical, not numbers.
    """
    df = log_df.copy()

    if required_columns is None:
        required_columns = numeric_columns

    if "gameDate" in df.columns:
        df["gameDate"] = pd.to_datetime(df["gameDate"])

    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    before = len(df)
    df = df.dropna(subset=[c for c in required_columns if c in df.columns])
    dropped = before - len(df)
    if dropped > 0:
        print(f"Dropped {dropped} rows with missing stats")

    return df.sort_values("gameDate").reset_index(drop=True) if "gameDate" in df.columns else df

## Section 5, checking the full pipeline works, skaters and goalies both

Before trusting this for anything real, worth checking pull, clean, and
cache all work correctly, and specifically that goalies come through
with their own stat fields, not just skater fields with blanks. Same
approach as the roster checks earlier, a handful of pass/fail checks,
not a full test suite.

In [ ]:
GOALIE_NUMERIC_COLUMNS = ["savePctg", "goalsAgainst", "shotsAgainst"]
GOALIE_REQUIRED_COLUMNS = ["decision", "savePctg", "goalsAgainst", "shotsAgainst"]

def check_game_log_pipeline(player_id, player_label, stat_columns, season=CURRENT_SEASON, required_columns=None):
    checks_passed = 0
    checks_total = 0

    def check(condition, description):
        nonlocal checks_passed, checks_total
        checks_total += 1
        if condition:
            checks_passed += 1
            print(f"[pass] {description}")
        else:
            print(f"[FAIL] {description}")

    print(f"\n--- Checking pipeline for {player_label} (id: {player_id}) ---")

    raw_log = get_game_log(player_id, season=season, force_refresh=True)
    check(len(raw_log) > 0, "raw game log is not empty")

    all_expected_cols = required_columns if required_columns is not None else stat_columns
    check(all(col in raw_log.columns for col in all_expected_cols),
          f"expected stat columns are present ({all_expected_cols})")

    cleaned_log = clean_game_log(raw_log, stat_columns, required_columns=required_columns)
    check(len(cleaned_log) > 0, "cleaned game log is not empty after cleaning")
    check(len(cleaned_log) <= len(raw_log), "cleaning didn't somehow add rows")

    for col in stat_columns:
        if col in cleaned_log.columns:
            check(pd.api.types.is_numeric_dtype(cleaned_log[col]) or cleaned_log[col].dtype == object,
                  f"'{col}' has a sane dtype after cleaning")

    if "gameDate" in cleaned_log.columns:
        check(cleaned_log["gameDate"].is_monotonic_increasing, "games are sorted by date")

    check(not cleaned_log.duplicated(subset=["gameDate"]).any() if "gameDate" in cleaned_log.columns else True,
          "no duplicate games in the log")

    cached_log = get_game_log(player_id, season=season, force_refresh=False)
    age = cache_age_hours(f"gamelog_{player_id}_{season}")
    check(age is not None and age < 0.1, "second pull loaded from cache, not a fresh request")
    check(len(cached_log) == len(raw_log), "cached data matches what was originally pulled")

    print(f"{checks_passed}/{checks_total} checks passed for {player_label}")
    return checks_passed == checks_total

Section 5 recap, a few real issues turned up while testing this end to
end, an empty pre-season pull crashing the cache, a cleaning step that
was silently dropping every goalie's data regardless of how many games
they'd played, and a stat column list that only covered a fraction of
what the league actually scores. All fixed and verified above. Good
reminder that testing against edge cases, an empty season, a goalie
with one appearance, is what actually catches this kind of thing,
testing only against a star player having a normal season wouldn't
have surfaced any of it.

## Section 6, mapping scoring abbreviations to actual API fields

The scoring config uses short codes, G, A, SOG, and so on, since that's
how leagues actually describe their rules. The data coming back from
the API uses full field names, goals, assists, shots. Something needs
to sit between the two.

Worth being upfront, not every abbreviation maps directly to a field.
Some are a clean 1 to 1, some need a small calculation, like turning a
decision letter into a win or loss, or saves into shotsAgainst minus
goalsAgainst. A couple, hits and blocks specifically, aren't in this
endpoint at all, those would need a different data source entirely,
already noted in future_directions.

In [ ]:
SKATER_FIELD_MAP = {
    "G":    "goals",
    "A":    "assists",
    "P":    "points",
    "+/-":  "plusMinus",
    "PIM":  "pim",
    "PPG":  "powerPlayGoals",
    "PPA":  "ppAssists",       # derived, powerPlayPoints minus powerPlayGoals
    "PPP":  "powerPlayPoints",
    "SHG":  "shorthandedGoals",
    "SHA":  "shAssists",        # derived, shorthandedPoints minus shorthandedGoals
    "SHP":  "shorthandedPoints",
    "GWG":  "gameWinningGoals",
    "SOG":  "shots",
    "SH%":  "shootingPctg",
    "HIT":  "hits",              # from boxscore
    "BLK":  "blockedShots",      # from boxscore
    "FW":   None,   # not available as a raw count, only a percentage exists
    "FL":   None,   # same issue
}

GOALIE_FIELD_MAP = {
    "SA":   "shotsAgainst",
    "SV":   None,   # derived, shotsAgainst minus goalsAgainst
    "GA":   "goalsAgainst",
    "SV%":  "savePctg",
    "SHO":  "shutouts",
    "W":    "decision",
    "L":    "decision",
    "GS":   "gamesStarted",
    "GAA":  None,   # rate stat across games, not meaningful per single game row
}

In [ ]:
GOALIE_NUMERIC_COLUMNS = ["savePctg", "goalsAgainst", "shotsAgainst"]
GOALIE_REQUIRED_COLUMNS = ["savePctg", "goalsAgainst", "shotsAgainst"]  # decision no longer required

def clean_goalie_log(log_df, numeric_columns):
    """
    Cleaning specifically for goalie game logs. Never requires 'decision'
    to be present, relief appearances legitimately have none, but still
    have real, scoreable stats. Missing decision becomes 'ND' rather
    than causing the row to be dropped.
    """
    df = clean_game_log(log_df, numeric_columns, required_columns=numeric_columns)
    df["decision"] = df["decision"].fillna("ND") if "decision" in df.columns else "ND"
    return df

In [ ]:
def apply_scoring(games_df, scoring_config, field_map):
    """
    Converts a scoring config, abbreviation based, into fantasy points per game,
    using field_map to translate abbreviations into actual column names.
    Handles the derived cases, W, L, SV, separately since they're not a
    direct column read.
    """
    points = pd.Series(0.0, index=games_df.index)
    unmapped = []

    for stat, weight in scoring_config.items():
        field = field_map.get(stat)

        if stat == "W":
            points += (games_df["decision"] == "W").astype(int) * weight
        elif stat == "L":
            points += (games_df["decision"] == "L").astype(int) * weight
        elif stat == "SV":
            points += (games_df["shotsAgainst"] - games_df["goalsAgainst"]) * weight
        elif field is None:
            unmapped.append(stat)
        else:
            points += games_df[field] * weight

    if unmapped:
        print(f"Warning, these scored categories aren't available from this data source yet, "
              f"and were skipped: {unmapped}")

    return points

In [ ]:
def get_relevant_columns(scoring_config, field_map):
    """
    Works out which raw columns are actually needed to score a given
    scoring config, based on field_map. Ties testing and cleaning
    directly to the league's real scoring rules, so nothing gets missed
    if the scoring config changes later.
    """
    columns = set()
    for stat in scoring_config:
        if stat in ("W", "L"):
            columns.add("decision")
        elif stat == "SV":
            columns.add("shotsAgainst")
            columns.add("goalsAgainst")
        else:
            field = field_map.get(stat)
            if field is not None:
                columns.add(field)
    return sorted(columns)

SKATER_STAT_COLUMNS = get_relevant_columns(skater_scoring, SKATER_FIELD_MAP)
GOALIE_REQUIRED_COLUMNS = get_relevant_columns(goalie_scoring, GOALIE_FIELD_MAP)
GOALIE_NUMERIC_COLUMNS = [c for c in GOALIE_REQUIRED_COLUMNS if c != "decision"]

print("Skater columns needed:", SKATER_STAT_COLUMNS)
print("Goalie required columns:", GOALIE_REQUIRED_COLUMNS)
print("Goalie numeric columns:", GOALIE_NUMERIC_COLUMNS)

## Section 6.5, building the full skater stat line

Turns out a few categories don't need a new data source at all, they're
derivable from fields the game log already gives us. Points already
splits into goals and assists, so the same logic works for powerplay
and shorthanded points, subtract the goals from the points and what's
left is assists in that situation.

Hits and blocked shots do need the boxscore pull from before.

Faceoffs won and lost are a real gap, the API only exposes a faceoff
win percentage, not raw counts, and there's no way to reconstruct wins
and losses from a percentage alone without knowing total faceoffs
taken. Flagged honestly in future_directions rather than faked.

In [ ]:
def get_boxscore(game_id, refresh_after_hours=24, force_refresh=False):
    """
    Pulls the boxscore for a single game, cached by game id. A finished
    game's boxscore never changes, so once cached, it's cached for good,
    the refresh window here mostly matters for a game that's still live.
    """
    key = f"boxscore_{game_id}"
    age = cache_age_hours(key)

    if not force_refresh and age is not None and age < refresh_after_hours:
        return load_from_cache(key)

    url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/boxscore"
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()

    # flatten playerByGameStats into one row per player for this game
    rows = []
    stats = data.get("playerByGameStats", {})
    for side in ["awayTeam", "homeTeam"]:
        for group in ["forwards", "defense", "goalies"]:
            for player in stats.get(side, {}).get(group, []):
                row = dict(player)
                row["gameId"] = game_id
                rows.append(row)

    boxscore_df = pd.DataFrame(rows)
    if not boxscore_df.empty:
        save_to_cache(boxscore_df, key)
    return boxscore_df

In [ ]:
def get_hits_and_blocks(player_id, game_log, pause=0.3):
    """
    For every game in a player's game log, pulls that game's boxscore
    (cached, reused for other players later) and extracts hits and
    blockedShots for this specific player. Returns a small DataFrame
    keyed by gameId, ready to merge onto the existing game log.
    """
    records = []

    for game_id in game_log["gameId"]:
        boxscore = get_boxscore(game_id)
        if boxscore.empty:
            continue

        player_row = boxscore[boxscore["playerId"] == player_id]
        if player_row.empty:
            continue

        records.append({
            "gameId": game_id,
            "hits": player_row.iloc[0].get("hits", 0),
            "blockedShots": player_row.iloc[0].get("blockedShots", 0),
        })
        time.sleep(pause)

    return pd.DataFrame(records)

In [ ]:
ALL_SKATER_RAW_FIELDS = [
    "goals", "assists", "points", "plusMinus", "pim",
    "powerPlayGoals", "powerPlayPoints",
    "shorthandedGoals", "shorthandedPoints",
    "gameWinningGoals", "shots", "shootingPctg",
]

def build_full_skater_log(player_id, season, include_hits_blocks=True):
    """
    Pulls a skater's game log and builds every stat category we can
    actually source, direct fields, derived fields, and hits/blocks
    from the boxscore. Returns one row per game with everything filled in.
    """
    log = get_game_log(player_id, season=season, force_refresh=False)

    if log.empty:
        return log  # hasn't played this season, nothing to build

    log = clean_game_log(log, ALL_SKATER_RAW_FIELDS)

    # derived categories, no extra api calls needed
    log["ppAssists"] = log["powerPlayPoints"] - log["powerPlayGoals"]
    log["shAssists"] = log["shorthandedPoints"] - log["shorthandedGoals"]

    if include_hits_blocks:
        hits_blocks = get_hits_and_blocks(player_id, log)
        log = log.merge(hits_blocks, on="gameId", how="left")

    return log

In [ ]:
skater_id = lookup_player_id("McDavid")
weegar_id = lookup_player_id("Weegar")
mcdavid_full = build_full_skater_log(skater_id, LAST_SEASON)
weegar_full = build_full_skater_log(weegar_id, LAST_SEASON)

mcdavid_full["fantasy_points"] = apply_scoring(mcdavid_full, skater_scoring, SKATER_FIELD_MAP)
weegar_full["fantasy_points"] = apply_scoring(weegar_full, skater_scoring, SKATER_FIELD_MAP)

print(f"McDavid, avg fantasy points: {mcdavid_full['fantasy_points'].mean():.2f}")
print(f"Mack Daddy, avg fantasy points: {weegar_full['fantasy_points'].mean():.2f}")

## Section 6.6, verifying scoring against known season totals

Two kinds of checks here. First, internal consistency, does points
actually equal goals plus assists in our own data, do the derived
powerplay and shorthanded assist numbers add back up correctly.
Second, external validation, do our summed totals for the season match
McDavid's actual, publicly known 2025-26 stats, 82 games, 48 goals, 90
assists, 138 points, +17, 44 PIM, 306 shots, 13 powerplay goals, 4
game-winning goals. If our pulled data doesn't match these, something
in the pull or cleaning step is wrong, not just the scoring math.

In [ ]:
def check_internal_consistency(log_df, label):
    checks_passed = 0
    checks_total = 0

    def check(condition, description):
        nonlocal checks_passed, checks_total
        checks_total += 1
        if condition:
            checks_passed += 1
            print(f"[pass] {description}")
        else:
            print(f"[FAIL] {description}")

    print(f"\n--- Internal consistency, {label} ---")

    check((log_df["goals"] + log_df["assists"] == log_df["points"]).all(),
          "goals plus assists equals points, every game")

    check((log_df["ppAssists"] + log_df["powerPlayGoals"] == log_df["powerPlayPoints"]).all(),
          "derived ppAssists plus powerPlayGoals equals powerPlayPoints, every game")

    check((log_df["shAssists"] + log_df["shorthandedGoals"] == log_df["shorthandedPoints"]).all(),
          "derived shAssists plus shorthandedGoals equals shorthandedPoints, every game")

    check((log_df["ppAssists"] >= 0).all(), "no negative derived powerplay assists")
    check((log_df["shAssists"] >= 0).all(), "no negative derived shorthanded assists")

    if "hits" in log_df.columns:
        check(log_df["hits"].notna().sum() > 0, "hits data actually came through from boxscore, not all null")
    if "blockedShots" in log_df.columns:
        check(log_df["blockedShots"].notna().sum() > 0, "blocked shots data actually came through, not all null")

    print(f"{checks_passed}/{checks_total} checks passed")

check_internal_consistency(mcdavid_full, "McDavid")

In [ ]:
def check_against_known_totals(log_df, known_totals, label):
    checks_passed = 0
    checks_total = 0

    def check(condition, description):
        nonlocal checks_passed, checks_total
        checks_total += 1
        if condition:
            checks_passed += 1
            print(f"[pass] {description}")
        else:
            print(f"[FAIL] {description}")

    print(f"\n--- Checking against known real-world totals, {label} ---")

    for stat, expected in known_totals.items():
        actual = log_df[stat].sum()
        check(actual == expected, f"{stat}, expected {expected}, got {actual}")

    print(f"{checks_passed}/{checks_total} checks passed")

mcdavid_known_totals = {
    "goals": 48,
    "assists": 90,
    "points": 138,
    "plusMinus": 17,
    "pim": 44,
    "shots": 306,
    "powerPlayGoals": 13,
    "gameWinningGoals": 4,
    "hits": 40,           # fill in from nhl.com/stats once you check
    "blockedShots": 30,   # same
}

check_against_known_totals(mcdavid_full, mcdavid_known_totals, "McDavid")
print(f"\nGames in our data: {len(mcdavid_full)}, actual games played: 82")

## Section 6.7, testing on random players

Everything so far has been tested on hand-picked players, McDavid, a
specific goalie, Weegar. Worth checking the pipeline holds up on
players we didn't choose, since edge cases tend to hide in the players
nobody thinks to test. This can't check against known real-world
totals, since we don't have those memorized for random players, but it
can confirm the pipeline runs clean and the internal consistency checks
still pass.

In [ ]:
def pick_random_player(position_filter, min_games=10, season=LAST_SEASON, max_attempts=10):
    """
    Picks a random player matching a position filter, skips anyone with
    too few games in the given season, a very short sample isn't a
    useful test of the normal pipeline, that's its own edge case,
    already covered separately with Brossoit earlier.
    """
    candidates = all_players[all_players["position"].isin(position_filter)]

    for _ in range(max_attempts):
        candidate = candidates.sample(1).iloc[0]
        log = get_game_log(candidate["player_id"], season=season, force_refresh=False)
        if len(log) >= min_games:
            print(f"Picked {candidate['name']} ({candidate['position']}), {len(log)} games")
            return candidate["player_id"], candidate["name"]
        print(f"Skipping {candidate['name']}, only {len(log)} games")

    raise RuntimeError(f"Couldn't find a player with at least {min_games} games after {max_attempts} attempts")

random_forward_id, random_forward_name = pick_random_player(["C", "L", "R"])
random_defenseman_id, random_defenseman_name = pick_random_player(["D"])
random_goalie_id, random_goalie_name = pick_random_player(["G"])

In [ ]:
random_forward_log = build_full_skater_log(random_forward_id, LAST_SEASON)
random_forward_log["fantasy_points"] = apply_scoring(random_forward_log, skater_scoring, SKATER_FIELD_MAP)
check_internal_consistency(random_forward_log, random_forward_name)

random_defenseman_log = build_full_skater_log(random_defenseman_id, LAST_SEASON)
random_defenseman_log["fantasy_points"] = apply_scoring(random_defenseman_log, skater_scoring, SKATER_FIELD_MAP)
check_internal_consistency(random_defenseman_log, random_defenseman_name)

goalie_ok = check_game_log_pipeline(
    random_goalie_id, random_goalie_name,
    GOALIE_NUMERIC_COLUMNS, season=LAST_SEASON,
    required_columns=GOALIE_REQUIRED_COLUMNS
)

print(f"\n{random_forward_name}: avg fantasy points {random_forward_log['fantasy_points'].mean():.2f}")
print(f"{random_defenseman_name}: avg fantasy points {random_defenseman_log['fantasy_points'].mean():.2f}")

In [ ]:
random_goalie_log = get_game_log(random_goalie_id, season=LAST_SEASON, force_refresh=False)
random_goalie_log = clean_goalie_log(random_goalie_log, GOALIE_NUMERIC_COLUMNS)
random_goalie_log["fantasy_points"] = apply_scoring(random_goalie_log, goalie_scoring, GOALIE_FIELD_MAP)

print(f"{random_goalie_name}: {len(random_goalie_log)} games, avg fantasy points: {random_goalie_log['fantasy_points'].mean():.2f}")
random_goalie_log[["gameDate", "decision", "savePctg", "goalsAgainst", "fantasy_points"]].head(4)

In [ ]:
def apply_scoring_breakdown(games_df, scoring_config, field_map):
    """
    Same logic as apply_scoring, but returns a DataFrame with one column
    per scored category showing its point contribution, plus a total
    column, instead of collapsing straight to a single number. Useful
    for checking that every category is actually contributing something,
    not just trusting the final total.
    """
    breakdown = pd.DataFrame(index=games_df.index)
    unmapped = []

    for stat, weight in scoring_config.items():
        field = field_map.get(stat)

        if stat == "W":
            breakdown[f"{stat}_pts"] = (games_df["decision"] == "W").astype(int) * weight
        elif stat == "L":
            breakdown[f"{stat}_pts"] = (games_df["decision"] == "L").astype(int) * weight
        elif stat == "SV":
            breakdown[f"{stat}_pts"] = (games_df["shotsAgainst"] - games_df["goalsAgainst"]) * weight
        elif field is None:
            unmapped.append(stat)
            breakdown[f"{stat}_pts"] = 0.0
        else:
            breakdown[f"{stat}_pts"] = games_df[field] * weight

    breakdown["total"] = breakdown.sum(axis=1)

    if unmapped:
        print(f"Note, these categories aren't available and are showing as 0, not actually scored: {unmapped}")

    return breakdown

In [ ]:
random_goalie_log = get_game_log(random_goalie_id, season=LAST_SEASON, force_refresh=False)
random_goalie_log = clean_goalie_log(random_goalie_log, GOALIE_NUMERIC_COLUMNS)

scoring_breakdown = apply_scoring_breakdown(random_goalie_log, goalie_scoring, GOALIE_FIELD_MAP)
random_goalie_log["fantasy_points"] = scoring_breakdown["total"]

print(f"{random_goalie_name}: {len(random_goalie_log)} games, avg fantasy points: {random_goalie_log['fantasy_points'].mean():.2f}")

detailed_view = pd.concat([
    random_goalie_log[["gameDate", "decision", "savePctg", "goalsAgainst"]],
    scoring_breakdown
], axis=1)

detailed_view.head(4)

In [ ]:
raw_goalie_log = get_game_log(random_goalie_id, season=LAST_SEASON, force_refresh=True)
missing_rows = raw_goalie_log[raw_goalie_log[GOALIE_REQUIRED_COLUMNS].isna().any(axis=1)]
missing_rows

In [ ]:
def get_player_total_points(player_id, name, position, season=LAST_SEASON):
    if position == "G":
        log = get_game_log(player_id, season=season, force_refresh=False)
        if log.empty:
            return 0, 0.0
        log = clean_goalie_log(log, GOALIE_NUMERIC_COLUMNS)
        log["fantasy_points"] = apply_scoring(log, goalie_scoring, GOALIE_FIELD_MAP)
    else:
        log = build_full_skater_log(player_id, season)
        if log.empty:
            return 0, 0.0
        log["fantasy_points"] = apply_scoring(log, skater_scoring, SKATER_FIELD_MAP)

    return len(log), log["fantasy_points"].sum()

In [ ]:
random_sample = all_players.sample(5)

print(f"{'Name':<25}{'Pos':<6}{'Team':<6}{'Games':<8}{'Total Points'}")
print("-" * 60)

for _, player in random_sample.iterrows():
    games, total = get_player_total_points(player["player_id"], player["name"], player["position"])
    print(f"{player['name']:<25}{player['position']:<6}{player['team']:<6}{games:<8}{total:.1f}")

Section 6 recap, scoring turned out to need more than a simple lookup
table. A few categories, PPA and SHA specifically, were derivable from
existing fields without any extra API calls. Hits and blocked shots
needed a separate pull entirely, from the boxscore endpoint rather than
the game log. Faceoffs won and lost turned out to not be available as
raw counts anywhere in this API, only as a percentage, honestly flagged
as a known gap rather than approximated. Relief appearances by goalies
needed special handling too, missing a decision doesn't mean bad data,
it means no decision was awarded, and the row needed to be kept, not
dropped. Verified against a real player's known season totals, and
against 5 randomly selected players checked manually against Yahoo,
all matching.

## Section 7, rolling expected value and volatility

This is the mathematical core of the whole model. For every player, two
numbers get tracked, not one, $\mu$, the expected fantasy points per
game, and $\sigma$, how much that varies game to game. Both matter, two
players can average the same points per game and be completely
different assets if one is consistent and the other swings wildly.

### Why not a simple average

A flat average over the last $N$ games treats every game as equally
important. That's not quite right, hockey performance is streaky, a
role change, an injury recovery, a hot line, all make recent games more
informative than a game from a month ago. So instead of a flat average,
this uses an exponentially weighted moving average, EWMA, more recent
games count more, older games count less, and the weight decays
smoothly rather than falling off a cliff at some arbitrary cutoff.

### The EWMA formula

For a sequence of fantasy point totals $x_1, x_2, \dots, x_t$, in
chronological order, the EWMA at time $t$ is defined recursively,

$$
\mu_t = \alpha x_t + (1 - \alpha)\mu_{t-1}
$$

where $\alpha \in (0, 1]$ controls how much weight the newest
observation gets. A higher $\alpha$ means the average reacts faster to
recent games, a lower $\alpha$ means it's smoother and more stable, but
slower to pick up on a real change in form.

Unrolled, this is equivalent to a weighted average of every game so
far, with weights that shrink geometrically going back in time,

$$
\mu_t = \alpha \sum_{i=0}^{t-1} (1 - \alpha)^i \, x_{t-i}
$$

normalized so the weights sum to 1. That normalization matters early
on, with only a handful of games, without it the estimate would be
biased toward zero, pandas handles this correction automatically, more
on that below.

### Choosing alpha via a half-life

$\alpha$ isn't very intuitive on its own, a more natural way to set it
is by choosing a half-life, $h$, the number of games after which a
game's weight has decayed to half its original influence. The
relationship is,

$$
\alpha = 1 - \left(\frac{1}{2}\right)^{1/h}
$$

For example, a half-life of 5 games means a game from 5 games ago
counts half as much as the most recent game, and a game from 10 games
ago counts a quarter as much. This is a much easier thing to reason
about and explain than a raw alpha value, and it's the number that'll
actually get tuned later if the model needs adjusting.

### Volatility, sigma, under the same weighting

Sigma uses the same exponential weighting, not a flat standard
deviation, for the same reason mu does, a player's recent volatility is
more relevant than volatility from two months ago. The exponentially
weighted variance follows a similar recursive form,

$$
\sigma_t^2 = \alpha (x_t - \mu_{t-1})^2 + (1 - \alpha)\sigma_{t-1}^2
$$

and $\sigma_t = \sqrt{\sigma_t^2}$. Pandas computes both $\mu$ and
$\sigma$ this way directly, using `.ewm()`, so this doesn't need to be
hand-rolled, just understood.

In [ ]:
def compute_ewma_mu_sigma(points_series, half_life=5):
    """
    Computes the exponentially weighted mean (mu) and standard deviation
    (sigma) of a fantasy points series, most recent games weighted most
    heavily. half_life is in games, not a raw alpha, since half-life is
    the more interpretable knob to tune.
    """
    ewm = points_series.ewm(halflife=half_life, adjust=True)
    mu = ewm.mean()
    sigma = ewm.std()
    return mu, sigma

# test on McDavid's already-built season
mcdavid_mu, mcdavid_sigma = compute_ewma_mu_sigma(mcdavid_full["fantasy_points"])

mcdavid_full["mu"] = mcdavid_mu
mcdavid_full["sigma"] = mcdavid_sigma

mcdavid_full[["gameDate", "fantasy_points", "mu", "sigma"]].tail(10)

In [ ]:
mcdavid_full[["gameDate", "fantasy_points", "mu", "sigma"]].head(10)

In [ ]:
raw_ewm = mcdavid_full["fantasy_points"].ewm(halflife=5, adjust=False).mean()
corrected_ewm = mcdavid_full["fantasy_points"].ewm(halflife=5, adjust=True).mean()

pd.DataFrame({
    "fantasy_points": mcdavid_full["fantasy_points"],
    "raw (adjust=False)": raw_ewm,
    "corrected (adjust=True)": corrected_ewm
}).head(10)

In [ ]:
comparison = pd.DataFrame({
    "fantasy_points": mcdavid_full["fantasy_points"],
    "raw (adjust=False)": raw_ewm,
    "corrected (adjust=True)": corrected_ewm,
    "gap": corrected_ewm - raw_ewm
})
comparison.tail(15)

A quick side by side check confirms this, adjust=False, the plain
recursive formula, and adjust=True, pandas' early-sample corrected
version, diverge noticeably in the first several games, at game 1 they
differ by about 0.02, but by roughly game 20 to 30 the gap has shrunk
to a fraction of a point, and by game 65 to 70 onward it's on the order
of 1e-5, effectively zero. The correction matters early, exactly when
there isn't much history to work with yet, and becomes irrelevant once
enough games have accumulated.

In [ ]:
mcdavid_full[mcdavid_full["fantasy_points"] < 0][["gameDate", "goals", "assists", "pim", "fantasy_points"]]

## Section 8, shrinkage toward a prior season

Early in a season, there aren't enough games to trust mu and sigma on
their own, game one alone gives no sigma at all, as seen already.
Shrinkage solves this by blending in last season's stats as a prior,
leaned on heavily early, fading out as real current-season games
accumulate.

### The blending formula

$$
\hat{\mu} = \frac{n \cdot \mu_{\text{current}} + k \cdot \mu_{\text{prior}}}{n + k}
$$

where $n$ is the number of current-season games observed so far, and
$k$ is a constant controlling how strongly the prior is trusted. A
larger $k$ means the prior dominates longer before current-season data
takes over, a smaller $k$ means current data takes over faster. The
same blend applies to the variance,

$$
\hat{\sigma}^2 = \frac{n \cdot \sigma^2_{\text{current}} + k \cdot \sigma^2_{\text{prior}}}{n + k}
$$

and $\hat{\sigma} = \sqrt{\hat{\sigma}^2}$.

Two limiting cases worth checking by hand, when $n = 0$, no current
season games yet, $\hat{\mu} = \mu_{\text{prior}}$ exactly, the
estimate is entirely the prior, which is correct, there's nothing else
to go on. As $n \to \infty$, the $k \cdot \mu_{\text{prior}}$ term
becomes negligible next to $n \cdot \mu_{\text{current}}$, and
$\hat{\mu} \to \mu_{\text{current}}$, the prior fades out entirely,
also correct.

### Testing this without a real current season

The actual current season, 2026-27, has zero games right now, so there
isn't real partial-season data to test blending against yet. To confirm
the mechanism actually works before the real season gives us genuine
partial data, this simulates it, treating 2024-25 as the prior and
2025-26 as if it were still in progress, checking the blend at a few
different points partway through that season.

In [ ]:
PRIOR_SEASON = "20242025"

def get_prior_season_stats(player_id, season, position):
    """
    Computes simple season-long mean and std from a completed season,
    used as the shrinkage prior. Not EWMA, the season is already over,
    so there's no recency to weight, just the overall level and spread.
    """
    if position == "G":
        log = get_game_log(player_id, season=season, force_refresh=False)
        if log.empty:
            return None, None
        log = clean_goalie_log(log, GOALIE_NUMERIC_COLUMNS)
        log["fantasy_points"] = apply_scoring(log, goalie_scoring, GOALIE_FIELD_MAP)
    else:
        log = build_full_skater_log(player_id, season)
        if log.empty:
            return None, None
        log["fantasy_points"] = apply_scoring(log, skater_scoring, SKATER_FIELD_MAP)

    return log["fantasy_points"].mean(), log["fantasy_points"].std()

def blend_mu_sigma(current_mu, current_sigma, prior_mu, prior_sigma, n, k=10):
    blended_mu = (n * current_mu + k * prior_mu) / (n + k)
    blended_var = (n * current_sigma**2 + k * prior_sigma**2) / (n + k)
    return blended_mu, blended_var**0.5



In [ ]:
prior_mu, prior_sigma = get_prior_season_stats(skater_id, PRIOR_SEASON, "C")
print(f"McDavid, 2024-25 prior, mu: {prior_mu:.2f}, sigma: {prior_sigma:.2f}")

for n_games in [1, 5, 10, 20, 40, 82]:
    partial_log = mcdavid_full.iloc[:n_games]
    current_mu = partial_log["fantasy_points"].mean()
    current_sigma = partial_log["fantasy_points"].std() if n_games > 1 else 0

    blended_mu, blended_sigma = blend_mu_sigma(current_mu, current_sigma, prior_mu, prior_sigma, n=n_games, k=10)

    print(f"n={n_games:>3}, current mu={current_mu:.2f}, blended mu={blended_mu:.2f}, "
          f"current sigma={current_sigma:.2f}, blended sigma={blended_sigma:.2f}")

In [ ]:
print((82 * 2.75 + 10 * 2.23) / 92)

## Visualizing k, how fast should the prior fade

Rather than picking k by intuition, worth actually seeing how different
values behave. A small k lets current-season data take over almost
immediately, a large k keeps leaning on last season for a long time.
Plotting a few options side by side against the actual game log makes
this a visible, informed choice, not a guess.

In [ ]:
import matplotlib.pyplot as plt

k_values = [2, 5, 10, 20, 40]
games_range = range(1, 83)

plt.figure(figsize=(10, 6))

# reference lines, the two limiting cases the formula should approach
plt.axhline(prior_mu, color="gray", linestyle="--", label="prior mu (2024-25)")
final_season_mu = mcdavid_full["fantasy_points"].mean()
plt.axhline(final_season_mu, color="black", linestyle=":", label="final current-season mu")

for k in k_values:
    blended_values = []
    for n_games in games_range:
        partial_log = mcdavid_full.iloc[:n_games]
        current_mu = partial_log["fantasy_points"].mean()
        current_sigma = partial_log["fantasy_points"].std() if n_games > 1 else 0
        blended_mu, _ = blend_mu_sigma(current_mu, current_sigma, prior_mu, prior_sigma, n=n_games, k=k)
        blended_values.append(blended_mu)
    plt.plot(games_range, blended_values, label=f"k = {k}")

plt.xlabel("Games played this season")
plt.ylabel("Blended mu")
plt.title("How different k values let go of the prior over the season")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## A tempting alternative, blending with EWMA instead

The natural next thought is to blend the prior with the EWMA mu from
Section 7 instead of the flat cumulative mean, one number that's both
recency-weighted and prior-informed. Tried this, and it didn't hold up,
details below.

## Correction, EWMA and shrinkage answer different questions

An earlier version of this notebook blended the prior with the EWMA
value, expecting it to be smoother than blending with a flat cumulative
mean. Plotting it showed the opposite, blending with EWMA stayed
volatile all season, never settling down. The reason, EWMA's alpha
stays constant regardless of how many games have been played, so it
never stops reacting to recent streaks, while a flat mean's sensitivity
to any single new game shrinks as 1/n, naturally stabilizing over the
season.

That means these two are actually answering different questions, EWMA
answers, how is this player doing right now, recently. A cumulative
mean answers, what is this player's underlying level this season,
overall. Shrinkage is a season-level question, so it belongs with the
flat mean, not EWMA. Going forward, both get computed and kept
separate, blended mu, for season-level talent estimates and
comparisons, and EWMA mu, for a recent hot or cold signal, used
differently, not combined into one number.

## Choosing k = 10

Comparing k values on real season data, k=2 overreacts to hot and cold
stretches that are ultimately just noise, spiking well above the
eventual season average around games 35-45. k=40 stays the most stable
but is noticeably slow to trust a real, sustained improvement, still
sitting below the final average even after a full 82 games. k=10 sits
in between, damping early noise while still converging reasonably close
to the final season mu by the end of the year.

This is a judgment call, not something derived mathematically, worth
revisiting once real weekly predictions accumulate this season, if
predictions consistently lag behind what's actually happening, that's
a sign k is too large, if they overreact to one good or bad week, k is
too small. Logged here as v1's choice, k=10, open to tuning based on
actual calibration once there's real data to check it against.